# SynthTextEval evaluation pipeline for ICL-PILOT

This notebook adapts the [SynthTextEval](https://github.com/kr-ramesh/synthtexteval) pipeline to the traceable ENNI-style narratives in ICL-PILOT. It evaluates generated versus real text with corpus-level descriptive and automated text-quality measures, downstream TD/SLI classification, and age-cohort performance auditing.

The classification implementation is supplementary: it uses SynthTextEval's BERT classifier and fairness utilities but preserves ICL-PILOT's rotating prompt/evaluation separation. It does not replace the project's existing analysis. Literal `TD` and `SLI` labels are retained throughout. A distinct, child-level Gillam export is prepared as a test-only external-transfer set; it is never appended to the ENNI source corpus.

The package's entity-privacy module is not applicable to these de-identified child-story texts: there are no protected identifiers to audit, and shared story characters would make entity overlap uninterpretable. Its coreference module is also out of scope because the export has no gold coreference annotation. The raw export is preserved; a known narrative-external LLM meta-response is excluded only from analytic inputs and recorded explicitly.

In [ ]:
# Repository paths and run configuration. In Colab, a fresh clone needs the notebook and
# all-story export committed to the repository before this cell is run.
from pathlib import Path
import sys

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
COLAB_REPO_DIR = Path("/content/ICL-PILOT")
candidate_roots = [COLAB_REPO_DIR, Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in candidate_roots if (path / "pyproject.toml").exists()), None)

if PROJECT_ROOT is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Run this notebook from the ICL-PILOT repository or open it in Colab.")
    !git clone {REPO_URL} {COLAB_REPO_DIR}
    PROJECT_ROOT = COLAB_REPO_DIR

%cd {PROJECT_ROOT}

TEXT_RECORDS_CSV = PROJECT_ROOT / "phase2/generated/all_ages/text/all_story_text_records_4_to_7.csv"
GILLAM_EXTERNAL_TEXT_CSV = PROJECT_ROOT / "phase2/Gillam/text/gillam_external_text_records.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs/synthtexteval_demo"
TEXT_COLUMN = "story_text"
SOURCE_COLUMN = "source"
GROUP_COLUMN = "group"
COHORT_COLUMN = "cohort_year"
COMPARISON_COHORTS = (4, 5, 6, 7)
COMPARISON_GROUPS = ("TD", "SLI")
SAMPLE_SEED = 42
MAX_SAMPLES_PER_GROUP = None  # Set to 20 for a CPU smoke test.
RUN_MAUVE = False  # Enable only in a GPU runtime after the other text-quality checks.
RUN_CLASSIFICATION = False  # Requires a CUDA-enabled Colab runtime; enable after checking the prepared splits.
RUN_GILLAM_EXTERNAL_TEST = True  # Used only when RUN_CLASSIFICATION is True; Gillam remains test-only.
CLASSIFICATION_ROUNDS = ("round_01", "round_02", "round_03", "round_04")
CLASSIFIER_MODEL = "bert-base-uncased"
LABEL_MAPPING = {"SLI": 0, "TD": 1}
META_RESPONSE_MARKERS = ("I replaced the main characters to match the target story",)

assert TEXT_RECORDS_CSV.exists(), f"Missing text record: {TEXT_RECORDS_CSV}"
assert GILLAM_EXTERNAL_TEXT_CSV.exists(), f"Missing Gillam external-test export: {GILLAM_EXTERNAL_TEXT_CSV}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print({"project_root": str(PROJECT_ROOT), "text_records": str(TEXT_RECORDS_CSV), "gillam_external_records": str(GILLAM_EXTERNAL_TEXT_CSV), "results_dir": str(RESULTS_DIR)})

In [ ]:
# SynthTextEval's source distribution is not currently published to PyPI.
# Install the inspected GitHub revision rather than the non-working PyPI version string.
%pip -q install "git+https://github.com/kr-ramesh/synthtexteval.git@3d63a172418a985fd0d997e5c993f6a0390c294e" pyLDAvis
!python -m spacy download en_core_web_sm -q

In [ ]:
import json
import pandas as pd

text_records = pd.read_csv(TEXT_RECORDS_CSV)
required_columns = {
    SOURCE_COLUMN, GROUP_COLUMN, COHORT_COLUMN, TEXT_COLUMN, "pair_id", "round_id",
    "source_child_id", "td_child_id", "sli_child_id",
}
missing_columns = required_columns.difference(text_records.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"
assert not text_records[TEXT_COLUMN].isna().any()
assert (text_records[TEXT_COLUMN].astype(str).str.strip() != "").all()

# Preserve the raw export and make the analytic exclusion reproducible. This excludes only
# the known LLM self-commentary, never human ENNI text.
meta_response_mask = pd.Series(False, index=text_records.index)
for marker in META_RESPONSE_MARKERS:
    meta_response_mask |= text_records[TEXT_COLUMN].str.contains(marker, case=False, regex=False, na=False)
meta_response_mask &= text_records[SOURCE_COLUMN].eq("synthetic")
excluded_records = text_records.loc[meta_response_mask].copy()
analysis_records = text_records.loc[~meta_response_mask].copy()
excluded_records.to_csv(RESULTS_DIR / "excluded_meta_generations.csv", index=False)

summary = {
    "raw_records": int(len(text_records)),
    "analytic_records": int(len(analysis_records)),
    "excluded_meta_generations": int(len(excluded_records)),
    "exclusion_markers": list(META_RESPONSE_MARKERS),
}
(RESULTS_DIR / "input_qc_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(summary)
display(excluded_records[["source_csv", "source_row_number", "pair_id", "story_id", "stage"]])

In [ ]:
# Balanced, non-paired corpus samples for SynthTextEval's descriptive and quality modules.
synthetic_parts, reference_parts = [], []
for cohort_year in COMPARISON_COHORTS:
    for group in COMPARISON_GROUPS:
        synthetic_group = analysis_records.loc[
            (analysis_records[SOURCE_COLUMN] == "synthetic")
            & (analysis_records[COHORT_COLUMN] == cohort_year)
            & (analysis_records[GROUP_COLUMN] == group),
            [TEXT_COLUMN, COHORT_COLUMN, GROUP_COLUMN],
        ]
        reference_group = analysis_records.loc[
            (analysis_records[SOURCE_COLUMN] == "real")
            & (analysis_records[COHORT_COLUMN] == cohort_year)
            & (analysis_records[GROUP_COLUMN] == group),
            [TEXT_COLUMN, COHORT_COLUMN, GROUP_COLUMN],
        ]
        n = min(len(synthetic_group), len(reference_group))
        if MAX_SAMPLES_PER_GROUP is not None:
            n = min(n, MAX_SAMPLES_PER_GROUP)
        assert n > 0, f"No comparable {cohort_year}-year-old {group} text records found."
        seed = SAMPLE_SEED + 100 * cohort_year + (0 if group == "TD" else 1)
        synthetic_parts.append(synthetic_group.sample(n=n, random_state=seed).reset_index(drop=True))
        reference_parts.append(reference_group.sample(n=n, random_state=seed).reset_index(drop=True))

synthetic_texts = pd.concat(synthetic_parts, ignore_index=True)
reference_texts = pd.concat(reference_parts, ignore_index=True)
comparison_df = pd.DataFrame({
    "source": synthetic_texts[TEXT_COLUMN].tolist(),
    "reference": reference_texts[TEXT_COLUMN].tolist(),
})
comparison_manifest = pd.DataFrame({
    "source_cohort_year": synthetic_texts[COHORT_COLUMN],
    "source_group": synthetic_texts[GROUP_COLUMN],
    "reference_cohort_year": reference_texts[COHORT_COLUMN],
    "reference_group": reference_texts[GROUP_COLUMN],
})
comparison_manifest.to_csv(RESULTS_DIR / "balanced_corpus_sample_manifest.csv", index=False)
print({"synthetic_texts": len(synthetic_texts), "reference_texts": len(reference_texts)})
display(comparison_manifest.value_counts().rename("n_texts").reset_index())

## Descriptive analysis

These are generic corpus-level comparisons. The matching is balanced by cohort and literal corpus group, but row order does not define a child-, prompt-, or story-level pair.

In [ ]:
import nltk
from synthtexteval.eval.descriptive.descriptor import TextDescriptor
from synthtexteval.eval.descriptive.arguments import TextDescriptorArgs

nltk.download("punkt_tab")
synth_df = comparison_df.rename(columns={"source": "output_text"})[["output_text"]].copy()
real_texts = comparison_df["reference"].tolist()
descriptor_args = TextDescriptorArgs(
    pkl_file_path=str(RESULTS_DIR / "entity-output.pkl"),
    plt_file_path=str(RESULTS_DIR / "entity-analysis.png"),
    produce_plot=True,
)
desc_analyze = TextDescriptor(
    texts=synth_df["output_text"].tolist(),
    args=descriptor_args,
    reference_texts=real_texts,
)
desc_analyze._compare_to_reference_distribution(metrics=["text-length", "jaccard", "cosine"])

# Optional interactive topic display.
import pyLDAvis
pyLDAvis.enable_notebook()
topic_model = desc_analyze._topic_modeling_display(num_topics=3)
topic_model

## Downstream utility: rotating synthetic-to-real TD/SLI classification

For each usable generation round, the notebook derives pair membership from the IDs embedded in the generated artifacts (including the four-year pairs). Synthetic descendants of prompt pairs and real prompt-pair narratives form the training material; one whole prompt pair per cohort is reserved for validation; real narratives from all other paired children form the test set. This avoids treating synthetic replicates as independent children and prevents a prompt child from entering the corresponding real test set.

The stock SynthTextEval random split helper is deliberately not used: it would split related prompt-pair descendants across partitions. The notebook still uses SynthTextEval's classifier and cohort-wise fairness utilities.

The same trained models can additionally be tested on Gillam as a strictly external corpus. Gillam contributes no training or validation rows, and an ID-disjointness assertion is enforced. Because each Gillam row is a complete child transcript whereas the internal ENNI analysis is story-level, this is an external-transfer stress test rather than a like-for-like internal classification estimate.

In [ ]:
# Build round-specific datasets from the generated-artifact IDs, not the conflicting 4-year roster mapping.
CLASSIFICATION_DIR = RESULTS_DIR / "classification"
CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)

synthetic_source = analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("synthetic")].copy()
real_source = analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("real")].copy()
for column in ("source_child_id", "td_child_id", "sli_child_id"):
    synthetic_source[column] = pd.to_numeric(synthetic_source[column], errors="raise").astype("int64")
real_source["source_child_id"] = pd.to_numeric(real_source["source_child_id"], errors="raise").astype("int64")

# Gillam is a separate external test source. Each row is one complete child transcript;
# it is never concatenated with analysis_records or any training frame.
gillam_records = pd.read_csv(GILLAM_EXTERNAL_TEXT_CSV)
gillam_required = {TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "source", "sex"}
gillam_missing = gillam_required.difference(gillam_records.columns)
assert not gillam_missing, f"Missing Gillam columns: {sorted(gillam_missing)}"
assert gillam_records["source"].eq("real_external").all()
assert set(gillam_records[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
assert gillam_records["source_child_id"].nunique() == len(gillam_records)
assert gillam_records[TEXT_COLUMN].notna().all()
gillam_ids = set(gillam_records["source_child_id"].astype(str))
enni_ids = set(real_source["source_child_id"].astype(str))
external_id_overlap = sorted(gillam_ids.intersection(enni_ids))
assert not external_id_overlap, f"Gillam external IDs occur in ENNI data: {external_id_overlap}"
external_gillam_test = gillam_records[[TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "source_dataset", "sex"]].copy()
external_gillam_test = external_gillam_test.rename(columns={TEXT_COLUMN: "output_text"})
external_gillam_test["Label"] = external_gillam_test[GROUP_COLUMN].map(LABEL_MAPPING)
external_gillam_test["classification_round"] = "external_gillam"
external_gillam_test["pair_id"] = pd.NA
assert external_gillam_test["Label"].notna().all()
EXTERNAL_GILLAM_DIR = CLASSIFICATION_DIR / "external_gillam"
EXTERNAL_GILLAM_DIR.mkdir(parents=True, exist_ok=True)
external_gillam_test.to_csv(EXTERNAL_GILLAM_DIR / "test.csv", index=False)
external_gillam_audit = {
    "status": "prepared_test_only", "rows": int(len(external_gillam_test)),
    "unique_children": int(external_gillam_test["source_child_id"].nunique()),
    "group_counts": external_gillam_test[GROUP_COLUMN].value_counts().to_dict(),
    "enni_id_overlap": len(external_id_overlap),
    "document_unit": "one complete Gillam child transcript",
    "training_use": False,
}
(EXTERNAL_GILLAM_DIR / "input_audit.json").write_text(json.dumps(external_gillam_audit, indent=2) + "\n")
print({"external_gillam": external_gillam_audit})

pair_metadata = synthetic_source[[COHORT_COLUMN, "pair_id", "td_child_id", "sli_child_id"]].drop_duplicates()
assert pair_metadata.groupby([COHORT_COLUMN, "pair_id"]).size().eq(1).all(), "Ambiguous generated pair metadata."
pair_rounds = synthetic_source[[COHORT_COLUMN, "pair_id", "round_id"]].drop_duplicates()
assert pair_rounds.groupby([COHORT_COLUMN, "pair_id"])["round_id"].nunique().eq(1).all(), "A generated pair occurs in multiple rounds."

sli_pair_children = pair_metadata[[COHORT_COLUMN, "pair_id", "sli_child_id"]].rename(columns={"sli_child_id": "source_child_id"})
sli_pair_children[GROUP_COLUMN] = "SLI"
td_pair_children = pair_metadata[[COHORT_COLUMN, "pair_id", "td_child_id"]].rename(columns={"td_child_id": "source_child_id"})
td_pair_children[GROUP_COLUMN] = "TD"
pair_children = pd.concat([sli_pair_children, td_pair_children], ignore_index=True)
assert not pair_children.duplicated([COHORT_COLUMN, "pair_id", GROUP_COLUMN]).any()

real_columns = [TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id"]
real_source = real_source[real_columns].copy()
def real_rows_for_pairs(pair_ids):
    pair_lookup = pair_children.loc[pair_children["pair_id"].isin(pair_ids)]
    rows = real_source.merge(
        pair_lookup, on=[COHORT_COLUMN, GROUP_COLUMN, "source_child_id"], how="inner", validate="many_to_one"
    )
    return rows

def classifier_frame(frame, classification_round):
    output = frame[[TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "pair_id"]].copy()
    output = output.rename(columns={TEXT_COLUMN: "output_text"})
    output["Label"] = output[GROUP_COLUMN].map(LABEL_MAPPING)
    output["classification_round"] = classification_round
    assert output["Label"].notna().all()
    return output

prepared_rounds, split_audit = {}, []
for round_id in CLASSIFICATION_ROUNDS:
    round_synthetic = synthetic_source.loc[synthetic_source["round_id"].eq(round_id)].copy()
    if round_synthetic.empty:
        split_audit.append({"round_id": round_id, "status": "skipped_no_generated_rows"})
        continue
    train_pair_ids, validation_pair_ids, test_pair_ids = set(), set(), set()
    usable_cohorts = []
    for cohort_year in sorted(round_synthetic[COHORT_COLUMN].unique()):
        prompt_pairs = sorted(round_synthetic.loc[round_synthetic[COHORT_COLUMN].eq(cohort_year), "pair_id"].unique())
        all_pairs = set(pair_metadata.loc[pair_metadata[COHORT_COLUMN].eq(cohort_year), "pair_id"])
        if len(prompt_pairs) < 2:
            split_audit.append({"round_id": round_id, "cohort_year": int(cohort_year), "status": "skipped_one_prompt_pair"})
            continue
        validation_pair = prompt_pairs[-1]  # Deterministic whole-pair validation split; no test data are used.
        train_pair_ids.update(set(prompt_pairs) - {validation_pair})
        validation_pair_ids.add(validation_pair)
        test_pair_ids.update(all_pairs - set(prompt_pairs))
        usable_cohorts.append(int(cohort_year))
    if not usable_cohorts:
        continue
    train_pairs = pair_metadata.loc[pair_metadata["pair_id"].isin(train_pair_ids)]
    validation_pairs = pair_metadata.loc[pair_metadata["pair_id"].isin(validation_pair_ids)]
    usable_pair_ids = set(train_pairs["pair_id"]) | set(validation_pairs["pair_id"])
    synthetic_train = round_synthetic.loc[round_synthetic["pair_id"].isin(set(train_pairs["pair_id"]))]
    real_train = real_rows_for_pairs(set(train_pairs["pair_id"]))
    real_validation = real_rows_for_pairs(set(validation_pairs["pair_id"]))
    real_test = real_rows_for_pairs(test_pair_ids)
    real_train = real_train.loc[real_train[COHORT_COLUMN].isin(usable_cohorts)]
    real_validation = real_validation.loc[real_validation[COHORT_COLUMN].isin(usable_cohorts)]
    real_test = real_test.loc[real_test[COHORT_COLUMN].isin(usable_cohorts)]
    test_children = set(zip(real_test[GROUP_COLUMN], real_test["source_child_id"]))
    prompt_children = set(zip(pair_children.loc[pair_children["pair_id"].isin(usable_pair_ids), GROUP_COLUMN], pair_children.loc[pair_children["pair_id"].isin(usable_pair_ids), "source_child_id"]))
    assert not test_children.intersection(prompt_children), f"Prompt/test child overlap in {round_id}."
    assert set(synthetic_train[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_train[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_validation[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_test[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    round_dir = CLASSIFICATION_DIR / round_id
    round_dir.mkdir(parents=True, exist_ok=True)
    frames = {
        "train": classifier_frame(real_train, round_id),
        "validation": classifier_frame(real_validation, round_id),
        "synthetic": classifier_frame(synthetic_train, round_id),
        "test": classifier_frame(real_test, round_id),
    }
    for split_name, frame in frames.items():
        frame.to_csv(round_dir / f"{split_name}.csv", index=False)
    (round_dir / "label_mapping.json").write_text(json.dumps(LABEL_MAPPING, indent=2) + "\n")
    prepared_rounds[round_id] = {"path": round_dir, **frames}
    split_audit.append({
        "round_id": round_id, "status": "prepared", "cohorts": usable_cohorts,
        "real_train_rows": len(frames["train"]), "real_validation_rows": len(frames["validation"]),
        "synthetic_train_rows": len(frames["synthetic"]), "real_test_rows": len(frames["test"]),
        "test_prompt_child_overlap": 0, "pair_mapping_source": "generated_artifact_metadata",
    })

split_audit_df = pd.DataFrame(split_audit)
split_audit_df.to_csv(CLASSIFICATION_DIR / "split_audit.csv", index=False)
display(split_audit_df)
assert prepared_rounds, "No usable round-specific classification datasets were prepared."

In [ ]:
# Fine-tune and test SynthTextEval classifiers. This package revision hard-codes CUDA,
# so this cell must run on a GPU runtime. It compares real-only, synthetic-only, and augmented training.
if RUN_CLASSIFICATION:
    import torch
    from synthtexteval.eval.downstream.classify.train_classifier import (
        Arguments, Classifier, ModelArguments, TrainingArguments,
    )

    assert torch.cuda.is_available(), "Enable a Colab GPU before running classifier training."
    prediction_frames = []
    training_settings = ("real-only", "synthetic-train-only", "synthetic-train-augment")
    for round_id, prepared in prepared_rounds.items():
        round_dir = prepared["path"]
        for training_setting in training_settings:
            setting_dir = round_dir / training_setting
            model_dir = setting_dir / "model"
            setting_dir.mkdir(parents=True, exist_ok=True)
            train_args = TrainingArguments(
                output_dir=str(setting_dir / "trainer"),
                num_train_epochs=3,
                per_device_train_batch_size=8,
                per_device_eval_batch_size=8,
                save_strategy="no",
                report_to=[],
            )
            model_args = ModelArguments(
                model_name=CLASSIFIER_MODEL,
                path_to_dataset=str(round_dir),
                path_to_model=str(model_dir),
                n_labels=len(LABEL_MAPPING),
                problem_type="single_label_classification",
                text_field="output_text",
                label_field="Label",
                synthetic_usage=training_setting,
                is_train=True,
            )
            Classifier(Arguments(train=train_args, model=model_args)).finetune_model()

            prediction_path = setting_dir / "test_predictions.csv"
            test_args = TrainingArguments(
                output_dir=str(setting_dir / "tester"),
                per_device_eval_batch_size=8,
                save_strategy="no",
                report_to=[],
            )
            test_model_args = ModelArguments(
                model_name=CLASSIFIER_MODEL,
                path_to_dataset=str(round_dir / "test.csv"),
                path_to_model=str(model_dir),
                path_to_output_csv=str(prediction_path),
                path_to_aggregated_results=str(setting_dir / "test_metrics.csv"),
                n_labels=len(LABEL_MAPPING),
                problem_type="single_label_classification",
                text_field="output_text",
                label_field="Label",
                retain_columns=["cohort_year", "classification_round", "group", "source_child_id", "pair_id"],
                is_test=True,
            )
            Classifier(Arguments(train=test_args, model=test_model_args)).test_model()
            prediction_frame = pd.read_csv(prediction_path)
            prediction_frame["training_setting"] = training_setting
            prediction_frame["evaluation_set"] = "internal_enni"
            prediction_frames.append(prediction_frame)

            if RUN_GILLAM_EXTERNAL_TEST:
                external_prediction_path = setting_dir / "external_gillam_predictions.csv"
                external_test_args = TrainingArguments(
                    output_dir=str(setting_dir / "external_gillam_tester"),
                    per_device_eval_batch_size=8,
                    save_strategy="no",
                    report_to=[],
                )
                external_model_args = ModelArguments(
                    model_name=CLASSIFIER_MODEL,
                    path_to_dataset=str(EXTERNAL_GILLAM_DIR / "test.csv"),
                    path_to_model=str(model_dir),
                    path_to_output_csv=str(external_prediction_path),
                    path_to_aggregated_results=str(setting_dir / "external_gillam_metrics.csv"),
                    n_labels=len(LABEL_MAPPING),
                    problem_type="single_label_classification",
                    text_field="output_text",
                    label_field="Label",
                    retain_columns=["cohort_year", "classification_round", "group", "source_child_id", "source_dataset", "sex"],
                    is_test=True,
                )
                Classifier(Arguments(train=external_test_args, model=external_model_args)).test_model()
                external_prediction_frame = pd.read_csv(external_prediction_path)
                external_prediction_frame["training_setting"] = training_setting
                external_prediction_frame["evaluation_set"] = "external_gillam"
                prediction_frames.append(external_prediction_frame)

    classification_predictions = pd.concat(prediction_frames, ignore_index=True)
    classification_predictions.to_csv(CLASSIFICATION_DIR / "all_round_predictions.csv", index=False)
    display(classification_predictions.groupby(["evaluation_set", "training_setting", "classification_round", "Ground"]).size().rename("n").reset_index())
else:
    print("Datasets prepared. Set RUN_CLASSIFICATION = True in a GPU runtime to train and test the models.")

In [ ]:
# SynthTextEval cohort-wise performance audit. Run after the classification cell.
if "classification_predictions" in globals():
    from synthtexteval.eval.fairness.metrics import analyze_group_fairness_performance

    cohort_audits = []
    internal_predictions = classification_predictions.loc[classification_predictions["evaluation_set"].eq("internal_enni")].copy()
    for training_setting, prediction_frame in internal_predictions.groupby("training_setting"):
        performance, fairness = analyze_group_fairness_performance(
            prediction_frame,
            problem_type="multiclass",
            num_classes=len(LABEL_MAPPING),
            subgroup_type="cohort_year",
        )
        performance["training_setting"] = training_setting
        fairness["training_setting"] = training_setting
        performance.to_csv(CLASSIFICATION_DIR / f"{training_setting}_cohort_performance.csv", index=False)
        fairness.to_csv(CLASSIFICATION_DIR / f"{training_setting}_cohort_fairness.csv", index=False)
        cohort_audits.append(performance)
    display(pd.concat(cohort_audits, ignore_index=True))
    if RUN_GILLAM_EXTERNAL_TEST:
        external_predictions = classification_predictions.loc[classification_predictions["evaluation_set"].eq("external_gillam")].copy()
        external_predictions.to_csv(CLASSIFICATION_DIR / "all_external_gillam_predictions.csv", index=False)
        display(external_predictions.groupby(["training_setting", "classification_round", "Ground"]).size().rename("n").reset_index())
else:
    print("Run the classifier cell first; no prediction file has been created.")

## Privacy and memorization

SynthTextEval's entity-leakage and canary utilities are intentionally not run: the stories contain no sensitive identifiers, and ENNI's shared character names and story events are not private entities. Any source-copying or content-fidelity analysis remains an explicitly separate project analysis rather than an invalid entity-privacy proxy.

## Automated text-quality evaluation

These toolkit outputs compare balanced synthetic and real corpora. They are not clinical validity measures. Perplexity and Fréchet distance can run on CPU; MAUVE should be enabled only in a GPU runtime.

In [ ]:
from synthtexteval.eval.text_quality.metrics import TextQualityEval
from synthtexteval.eval.text_quality.arguments import Arguments as TextQualityArguments, FrechetArgs, LMArgs, MauveArgs

quality_args = TextQualityArguments(
    frechet=FrechetArgs(source_text_column="source", ref_text_column="reference"),
    mauve=MauveArgs(source_text_column="source", ref_text_column="reference"),
    perplexity=LMArgs(source_text_column="source", ref_text_column="reference"),
)
quality_evaluator = TextQualityEval(quality_args)
quality_evaluator.calculate_perplexity(comparison_df)
quality_evaluator.calculate_fid_score(comparison_df)
if RUN_MAUVE:
    quality_evaluator.calculate_mauve_score(comparison_df)
quality_results = quality_evaluator.return_results()
quality_evaluator.print_metrics(quality_results)
(RESULTS_DIR / "text_quality_results.json").write_text(json.dumps(quality_results, indent=2, default=str) + "\n")